# RDD2022 India training and measurement

Run these cells on a Colab or Kaggle GPU runtime. The dataset download is about 502 MB, and full training is intentionally left to the hosted GPU.

In [ ]:
!git clone <REPOSITORY_URL> road-damage-detector
%cd road-damage-detector

In [ ]:
%pip install -r requirements.txt
%pip install ultralytics

## Prepare the dataset

The mirror export is downloaded separately with `download_rdd2022_mirror.py`. Preparation normalizes its `train`/`valid` YOLO folders into the repository layout and selects the darkest validation images as `val_night`.

In [ ]:
!python download_rdd2022_mirror.py --workspace YOUR_WORKSPACE --project YOUR_PROJECT --version YOUR_VERSION --location data/rdd2022
!python prepare_dataset.py --source-dir data/rdd2022

In [ ]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path('data/rdd2022/rdd2022.yaml').read_text())
assert config['train'] == 'images/train'
assert config['val'] == 'images/val'
train = {p.stem for p in Path('data/images/train').glob('*') if p.is_file()}
val = {p.stem for p in Path('data/images/val').glob('*') if p.is_file()}
assert train.isdisjoint(val), 'Train/validation leakage detected'
print(f'train images: {len(train)}; val images: {len(val)}; val_night images: {len(list(Path("data/images/val_night").glob("*")))}')

## Train

This produces the checkpoint used by the measurement cells.

In [ ]:
!python train.py --data data/rdd2022/rdd2022.yaml --epochs 100 --batch 8 --imgsz 640 --device 0

## Evaluate overall and night validation splits

The second command appends a separate headed run to the same Markdown report.

In [ ]:
CHECKPOINT = 'runs/road_damage/rdd2022_india/weights/best.pt'
!python eval.py --weights $CHECKPOINT --data data/rdd2022/rdd2022.yaml --output RESULTS.md
!python eval.py --weights $CHECKPOINT --data data/rdd2022/rdd2022_night.yaml --output RESULTS.md --append

In [ ]:
from pathlib import Path
sample_images = sorted(Path('data/images/val').glob('*'))
assert sample_images, 'No validation images were prepared'
sample_image = sample_images[0]
print(sample_image)
!python benchmark.py --weights $CHECKPOINT --image {sample_image} --device 0 --output BENCHMARK.md